# 🚀 Training YOLOv8l (Large) — Dataset: wisard_ir

Notebook ini melatih model **YOLOv8l** menggunakan dataset **wisard_ir** pada GPU **T4** di Google Colab.

### ✨ Fitur
- **Auto-resume**: Jika runtime terputus, training otomatis dilanjutkan dari checkpoint terakhir
- **Checkpoint setiap 20 epoch**: Meminimalkan kehilangan progres
- **Fast I/O**: Dataset di-extract dari ZIP ke disk lokal Colab

## ⚠️ Catatan VRAM
YOLOv8l adalah model besar. Pada T4 (16GB VRAM), batch size dibatasi ke **4**.
Jika tetap OOM, coba kurangi batch menjadi **2**.

## Persiapan
1. Pastikan Runtime diatur ke **GPU T4**
2. Pastikan `wisard_ir.zip` atau folder `wisard_ir/` ada di Google Drive
3. Jalankan semua cell secara berurutan
4. **Jika runtime terputus**: Jalankan ulang SEMUA cell — training otomatis dilanjutkan

## 1️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2️⃣ Install Dependencies

In [ ]:
!pip install -q ultralytics

## 3️⃣ Cek GPU & Setup Environment

In [ ]:
import os
import shutil
import torch
import yaml
import time
import zipfile

print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.version.cuda}")
print(f"GPU tersedia: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB" if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"cuDNN       : {torch.backends.cudnn.version()}")
else:
    raise RuntimeError("❌ GPU tidak tersedia! Pastikan Runtime diatur ke GPU T4.")

## 4️⃣ Copy Dataset ke Disk Lokal (ZIP → Extract)

In [ ]:
MODEL_VARIANT = 'yolov8l'
DATASET_NAME  = 'wisard_ir'
RUN_NAME      = f'{MODEL_VARIANT}_{DATASET_NAME}'

DRIVE_ROOT    = '/content/drive/MyDrive/drone-wisard'
DRIVE_DATASET = os.path.join(DRIVE_ROOT, 'datasets', DATASET_NAME)
DRIVE_ZIP_1   = os.path.join(DRIVE_ROOT, 'datasets', f'{DATASET_NAME}.zip')
DRIVE_ZIP_2   = os.path.join(DRIVE_ROOT, f'{DATASET_NAME}.zip')

LOCAL_BASE    = '/content/datasets'
LOCAL_DATASET = os.path.join(LOCAL_BASE, DATASET_NAME)
LOCAL_ZIP     = f'/content/{DATASET_NAME}.zip'

PROJECT_PATH  = os.path.join(DRIVE_ROOT, 'runs_optimized')
COLLECTED_DIR = os.path.join(PROJECT_PATH, 'collected_best_models')

if os.path.exists(os.path.join(LOCAL_DATASET, 'images', 'train')):
    n_local = len(os.listdir(os.path.join(LOCAL_DATASET, 'images', 'train')))
    print(f"✅ Dataset sudah ada di lokal ({n_local} file). Skip copy.")
else:
    os.makedirs(LOCAL_BASE, exist_ok=True)
    zip_source = None
    if os.path.exists(DRIVE_ZIP_1):
        zip_source = DRIVE_ZIP_1
    elif os.path.exists(DRIVE_ZIP_2):
        zip_source = DRIVE_ZIP_2

    if zip_source:
        zip_size_gb = os.path.getsize(zip_source) / (1024**3)
        print(f"📦 Ditemukan ZIP: {zip_source} ({zip_size_gb:.1f} GB)")
        print(f"⏳ Step 1/2: Copy ZIP ke disk lokal...")
        start = time.time()
        shutil.copy2(zip_source, LOCAL_ZIP)
        print(f"   ✅ Selesai dalam {time.time()-start:.0f} detik")
        print(f"⏳ Step 2/2: Extract ZIP di disk lokal...")
        start = time.time()
        with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
            zf.extractall(LOCAL_BASE)
        print(f"   ✅ Selesai dalam {time.time()-start:.0f} detik")
        os.remove(LOCAL_ZIP)
        if not os.path.exists(os.path.join(LOCAL_DATASET, 'images')):
            for item in os.listdir(LOCAL_BASE):
                if os.path.isdir(os.path.join(LOCAL_BASE, item, 'images')):
                    actual_path = os.path.join(LOCAL_BASE, item)
                    if actual_path != LOCAL_DATASET:
                        os.rename(actual_path, LOCAL_DATASET)
                    break
    elif os.path.exists(DRIVE_DATASET):
        print(f"⚠️ ZIP tidak ditemukan, copy folder dari Drive (lebih lambat)...")
        start = time.time()
        shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
        print(f"   ✅ Selesai dalam {time.time()-start:.0f} detik")
    else:
        raise FileNotFoundError(f"❌ Dataset tidak ditemukan! Cek: {DRIVE_ZIP_1} atau {DRIVE_DATASET}/")

DATASET_PATH = LOCAL_DATASET
DATA_YAML    = os.path.join(DATASET_PATH, 'data.yaml')
assert os.path.exists(DATA_YAML), f"❌ data.yaml tidak ditemukan di {DATA_YAML}"

with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = DATASET_PATH
data_cfg['train'] = 'images/train'
data_cfg['val']   = 'images/val'
data_cfg['test']  = 'images/test'
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

train_img_dir = os.path.join(DATASET_PATH, 'images', 'train')
n_train = len([f for f in os.listdir(train_img_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'))])
print(f"\n{'='*50}")
print(f"✅ Dataset path   : {DATASET_PATH} (LOKAL SSD)")
print(f"📷 Jumlah gambar train: {n_train}")
print(f"📁 Output training: {os.path.join(PROJECT_PATH, RUN_NAME)} (Google Drive)")
print(f"{'='*50}")
if n_train == 0:
    raise FileNotFoundError(f"❌ Folder {train_img_dir} kosong!")

## 5️⃣ Training YOLOv8l (dengan Auto-Resume)

| Parameter | Nilai |
|-----------|-------|
| Model | YOLOv8l (Large) |
| Epochs | 400 |
| Image Size | 640 |
| Batch Size | 4 (disesuaikan T4 16GB) |
| Patience | 100 |

> ⚠️ Jika terjadi OOM, kurangi `batch` menjadi **2**

> 🔄 **Auto-Resume**: Jika `last.pt` ditemukan, training otomatis dilanjutkan

In [ ]:
from ultralytics import YOLO

torch.cuda.empty_cache()

LAST_PT = os.path.join(PROJECT_PATH, RUN_NAME, 'weights', 'last.pt')
RESUME_TRAINING = os.path.exists(LAST_PT)

if RESUME_TRAINING:
    print("="*50)
    print("🔄 RESUME MODE — Melanjutkan training dari checkpoint terakhir")
    print(f"   Checkpoint: {LAST_PT}")
    print("="*50)
    model = YOLO(LAST_PT)
    results = model.train(resume=True)
else:
    print("="*50)
    print("🆕 FRESH START — Memulai training dari awal")
    print("="*50)
    model = YOLO(f'{MODEL_VARIANT}.pt')

    results = model.train(
        data=DATA_YAML,
        epochs=400, imgsz=640, batch=4, patience=100,
        device=0, workers=2, val=True, amp=True, deterministic=True,
        project=PROJECT_PATH, name=RUN_NAME, exist_ok=True,
        save=True, save_period=20, plots=True,
        lr0=0.01, lrf=0.01, cos_lr=True, warmup_epochs=5.0,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.1,
        degrees=15.0, translate=0.2, scale=0.5,
        flipud=0.5, fliplr=0.5, mosaic=1.0,
        mixup=0.1, copy_paste=0.1, erasing=0.4,
    )

print("\n" + "="*50)
print(f"✅ Training {MODEL_VARIANT.upper()} selesai!")
print("="*50)

## 6️⃣ Salin Best Model ke Folder Terpusat

In [ ]:
best_src = os.path.join(PROJECT_PATH, RUN_NAME, 'weights', 'best.pt')
os.makedirs(COLLECTED_DIR, exist_ok=True)
best_dst = os.path.join(COLLECTED_DIR, f'best_{RUN_NAME}.pt')

if os.path.exists(best_src):
    shutil.copy2(best_src, best_dst)
    print(f"✅ best.pt berhasil disalin ke: {best_dst}")
else:
    print(f"⚠️ WARNING: best.pt tidak ditemukan di {best_src}")

if os.path.exists(best_dst):
    print(f"📦 Ukuran model: {os.path.getsize(best_dst)/(1024*1024):.1f} MB")

## 7️⃣ Tampilkan Hasil Training

In [ ]:
from IPython.display import Image, display

results_dir = os.path.join(PROJECT_PATH, RUN_NAME)
plots = ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
         'P_curve.png', 'R_curve.png', 'F1_curve.png', 'PR_curve.png']

for plot in plots:
    plot_path = os.path.join(results_dir, plot)
    if os.path.exists(plot_path):
        print(f"\n📈 {plot}:")
        display(Image(filename=plot_path, width=800))